In [1]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
import operator
import requests
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("GROQ_API_KEY")

c:\Users\mypci\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [2]:
llm = ChatGroq(model="openai/gpt-oss-20b", api_key=api_key)

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city, "count": 1}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if data.get('results'):
            lat = data['results'][0]['latitude']
            lon = data['results'][0]['longitude']
        else:
            return f"City '{city}' not found."
    
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return f"Weather data for {city}: {str(data['current_weather'])}"
    return "Failed to get weather."

weather_tools = [get_weather]
weather_llm = llm.bind_tools(weather_tools)

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    next: str

In [4]:
def orchestrator(state: AgentState):
    system_prompt = """You are an orchestrator. Based on the user's question decide which agent to call.
    If the question is about weather, respond with exactly: WEATHER
    If the question is about anything else, respond with exactly: GENERAL
    Respond with only one word."""
    
    user_messages = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    messages = [SystemMessage(content=system_prompt)] + user_messages
    response = llm.invoke(messages)
    
    return {"next": response.content.strip()}

def weather_agent(state: AgentState):
    system_prompt = "You are a weather assistant. Use the get_weather tool to answer weather questions."
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    
    while True:
        response = weather_llm.invoke(messages)
        messages.append(response)
        
        if response.tool_calls:
            tool_results = ToolNode(weather_tools).invoke({"messages": messages})
            messages.extend(tool_results["messages"])
        else:
            return {"messages": [response]}

def general_agent(state: AgentState):
    system_prompt = "You are a helpful general assistant."
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

In [5]:
def route(state: AgentState):
    if state["next"] == "WEATHER":
        return "weather"
    return "general"

graph = StateGraph(AgentState)

graph.add_node("orchestrator", orchestrator)
graph.add_node("weather", weather_agent)
graph.add_node("general", general_agent)

graph.set_entry_point("orchestrator")
graph.add_conditional_edges("orchestrator", route)
graph.add_edge("weather", END)
graph.add_edge("general", END)

app = graph.compile()

In [6]:
result = app.invoke({"messages": [HumanMessage(content="What is the weather in Lahore?")], "next": ""})
print(result["messages"][-1].content)

**Lahore – Current Weather (as of 08:15 UTC, 2 Jul 2026)**  

- **Temperature:** 33.8 °C  
- **Wind:** 4.5 km/h from the southeast (≈ 137°)  
- **Sky:** Clear (weather code 0)  
- **Daytime:** Yes (sunny)  

If you need a forecast for the rest of the day or tomorrow, just let me know!


In [7]:
conversation = []

while True:
    user_input = input("User: ").strip()
    if user_input.lower() in ["exit", "quit"]:
        break
    conversation.append(HumanMessage(content=user_input))
    result = app.invoke({"messages": conversation, "next": ""})
    reply = result["messages"][-1]
    conversation.append(reply)
    print("Agent:", reply.content)

Agent: **Karachi – Current Weather (as of 08:30 UTC, 2 July 2026)**  

- **Temperature:** 31.9 °C  
- **Wind:** 16.5 km/h from the south‑west (≈ 224°)  
- **Sky:** Cloudy  
- **Daytime:** Yes (sunny but overcast)

It’s a warm, slightly windy day with plenty of cloud cover. Stay hydrated if you’re heading outdoors!
Agent: No – Lahore is actually warmer.  
- **Karachi:** ~31.9 °C  
- **Lahore:** ~33.9 °C  

So Lahore is about 2 °C hotter than Karachi right now.
Agent: Paris is the capital city of France.  
- **Country:** France  
- **Region:** Île-de-France (the Parisian region)  
- **Latitude/Longitude:** 48.8566° N, 2.3522° E  
- **Geographical setting:** It sits on the River Seine in the north‑central part of France, roughly 120 km (75 mi) east of the English Channel.  

So Paris is located in north‑central France, within the Île‑de‑France region.
Agent: I’m not sure what you’re asking about “exot.” Could you clarify what you’d like to know? For example, are you looking for exotic tra